In [4]:
import glob

import ipywidgets as widgets
from IPython.display import display, Image

path_form = widgets.Text(value='camera/some_dataset/xy/*.jpg', description="Path",
                         layout=widgets.Layout(flex='0 1 auto', width='auto'))
                        # layout=widgets.Layout(flex='0 1 auto', height='100px', min_height='100px', width='auto'))

_options = ["cameraフレーム番号", "変更日"]
sort_options = widgets.Dropdown(options=_options, description="Sort")

sort_reverse_checkbox = widgets.Checkbox(value=False, description="Reverse")

# ペジネーション用ウィジェット
items_per_page_dropdown = widgets.Dropdown(
    options=[5, 10, 20, 50],
    value=10,
    description="Items/Page"
)

prev_button = widgets.Button(description="◀ Prev", button_style='info')
next_button = widgets.Button(description="Next ▶", button_style='info')

current_page_inttext = widgets.IntText(
    value=1,
    description="Page",
    min=1
)

total_pages_label = widgets.Label(value="/ 1")

# グリッドレイアウトを作成（3列で表示）
# columns の数を変更すると列数が変わります
grid = widgets.Box([], layout=widgets.Layout(
    flex_flow='row wrap',  # 要素を水平に並べ、入りきらなくなったら折り返す
    align_items='flex-start',
    width='100%',  # 親コンテナの幅に合わせる
))

In [5]:
import os.path
from pathlib import Path
import math
import os
import datetime
import time

sort_key = None
all_image_paths = []  # 全画像パスを保持


def update_grid():
    """現在のページに応じてグリッドを更新"""
    items_per_page = items_per_page_dropdown.value
    current_page = current_page_inttext.value

    # 総ページ数を計算
    total_items = len(all_image_paths)
    total_pages = math.ceil(total_items / items_per_page) if total_items > 0 else 1

    # ページ番号が範囲外の場合は調整
    if current_page > total_pages:
        current_page = total_pages
        current_page_inttext.value = current_page
    if current_page < 1:
        current_page = 1
        current_page_inttext.value = current_page

    # 総ページ数ラベルを更新
    total_pages_label.value = f"/ {total_pages}"

    # 現在のページに表示する画像パスを取得
    start_idx = (current_page - 1) * items_per_page
    end_idx = start_idx + items_per_page
    page_image_paths = all_image_paths[start_idx:end_idx]

    items = []
    for path in page_image_paths:
        # 画像ファイルを読み込み
        img = Image(path, width=200)
        # 画像をウィジェットとして格納
        items.append(widgets.VBox([
            widgets.Label(value=path.name),
            # widgets.Label(value=str(path)),
            widgets.Label(value=convert_epoch_to_datetime_string(os.path.getmtime(path))),
            widgets.Image(value=open(path, 'rb').read(), format='jpg', width=200),
        ]))

    grid.children = items


def convert_epoch_to_datetime_string(mtime_epoch):
    """
    os.path.getmtime() が返す UNIX Epoch秒を YYYY/mm/ddThh:mm:ss.zzz 形式の
    文字列に変換します。
    """
    try:

        # 整数部と小数部に分割
        # 整数部: 秒 (datetime.fromtimestamp() に使用)
        # 小数部: ミリ秒部分 (.zzz の部分)
        seconds = int(mtime_epoch)
        # 小数部分をミリ秒に変換し、整数にする (例: 0.12345 -> 123)
        milliseconds = int(round((mtime_epoch - seconds) * 1000))
        # ただし、datetimeオブジェクトはマイクロ秒までを持つため、より正確な方法として
        # mtime_epoch全体を fromtimestamp に渡すのが一般的です。

        # datetime.fromtimestamp() を使用して datetime オブジェクトに変換
        # fromtimestamp は浮動小数点数を受け入れ、マイクロ秒までを処理します。
        dt_object = datetime.datetime.fromtimestamp(mtime_epoch)

        # strftime() で YYYY/mm/ddThh:mm:ss 形式に変換
        # %Y:年, %m:月, %d:日, %H:時, %M:分, %S:秒
        formatted_date = dt_object.strftime("%Y/%m/%dT%H:%M:%S")

        # マイクロ秒からミリ秒部分 (.zzz) を抽出し、文字列に追加
        # dt_object.microsecond は 0 から 999999 の値を取ります
        # これをミリ秒 (000 から 999) に変換し、3桁ゼロ埋めします
        milliseconds_str = f"{dt_object.microsecond // 1000:03d}"

        # 最終的な形式の文字列を構築
        result = f"{formatted_date}.{milliseconds_str}"

        return result

    except Exception as e:
        return f"エラーが発生しました: {e}"


def on_path_form_value_changed(change):
    global all_image_paths

    # 表示したい画像ファイルのパスを取得（例：'images'フォルダ内のjpgファイルすべて）
    all_image_paths = sorted(
        (p for x in glob.glob(change["new"])
         if (p := Path(x)).is_file() and x.endswith(".jpg")),
        key=sort_key,
        reverse=sort_reverse_checkbox.value,
    )

    # ページ番号を1にリセット
    current_page_inttext.value = 1

    # グリッドを更新
    update_grid()


path_form.observe(on_path_form_value_changed, names='value')


def on_sort_options_value_changed(change):
    global sort_key

    if change["new"] == _options[0]:
        # ファイル名が "some_path/<pwm1>_<pwm2>_<frame>.jpg" の形式であると仮定する
        sort_key = lambda x: int(x.stem.split("_")[-1])
    elif change["new"] == _options[1]:
        sort_key = os.path.getmtime
    else:
        sort_key = None
        print("Unknown option: {change['new']}")
        Z
    on_path_form_value_changed({"new": path_form.value})


sort_options.observe(on_sort_options_value_changed, names='value')

sort_reverse_checkbox.observe(lambda change: on_path_form_value_changed({"new": path_form.value}), names='value')


# ペジネーション機能のイベントハンドラー
def on_prev_button_clicked(b):
    if current_page_inttext.value > 1:
        current_page_inttext.value -= 1


def on_next_button_clicked(b):
    items_per_page = items_per_page_dropdown.value
    total_pages = math.ceil(len(all_image_paths) / items_per_page) if len(all_image_paths) > 0 else 1
    if current_page_inttext.value < total_pages:
        current_page_inttext.value += 1


def on_current_page_changed(change):
    update_grid()


def on_items_per_page_changed(change):
    # ページ番号を1にリセット
    current_page_inttext.value = 1
    update_grid()


prev_button.on_click(on_prev_button_clicked)
next_button.on_click(on_next_button_clicked)
current_page_inttext.observe(on_current_page_changed, names='value')
items_per_page_dropdown.observe(on_items_per_page_changed, names='value')

In [6]:
container = widgets.VBox([
    path_form,
    widgets.HBox([sort_options, sort_reverse_checkbox]),
    items_per_page_dropdown,
    widgets.HBox([prev_button, current_page_inttext, total_pages_label, next_button]),
    grid,
])

# グリッドを表示
display(container)